# Website Category Classifier

**Language-agnostic classification of websites from HTML source code**

This notebook demonstrates how to train and use a fastText-based classifier that categorizes websites based on their homepage HTML.

## Features
- **Language agnostic**: Works with any language (English, Czech, German, etc.)
- **Fast inference**: Uses fastText for sub-millisecond predictions
- **Multiclass support**: Classify into predefined categories
- **Production-ready**: Includes quantized models for deployment

## Supported Categories
- Adult, Automotive, Computers, Entertainment, Finance
- Food, Health, News, Shopping, Sports, Travel

In [ ]:
# Install dependencies if needed
%pip install -q pandas selectolax fasttext-wheel tqdm scikit-learn

In [ ]:
import os
import re
import json
import gzip
import base64
import html as ihtml
from pathlib import Path

import fasttext
import pandas as pd
from selectolax.parser import HTMLParser
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

tqdm.pandas()

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

## Configuration

In [ ]:
# Define your categories here
CATEGORIES = [
    "Adult",
    "Automotive",
    "Computers",
    "Entertainment",
    "Finance",
    "Food",
    "Health",
    "News",
    "Shopping",
    "Sports",
    "Travel",
]

# Paths
DATA_PATH = Path("../dataset.parquet")  # or .jsonl, .csv
MODEL_OUTPUT_DIR = Path("../model_output")
MODEL_PATH = MODEL_OUTPUT_DIR / "website_classifier.bin"

# Training parameters
TRAIN_EPOCHS = 20
TRAIN_LR = 0.5
TRAIN_DIM = 64

## Text Processing Functions

In [ ]:
# Regex patterns
RE_WS = re.compile(r"\s+")
RE_BAD_CTRL = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")
RE_LABEL_SAFE = re.compile(r"[^A-Za-z0-9]+")


def normalize_text(text: str | None) -> str:
    """Clean and normalize text."""
    if not text:
        return ""
    text = ihtml.unescape(str(text))
    text = RE_BAD_CTRL.sub(" ", text)
    text = RE_WS.sub(" ", text).strip()
    return text


def safe_label(label: str, valid_categories: set[str]) -> str:
    """Convert label to fastText-safe format."""
    label = label.strip()
    if label not in valid_categories:
        raise ValueError(f"Unknown label '{label}'. Valid: {valid_categories}")
    return f"__label__{RE_LABEL_SAFE.sub('_', label)}"


def unsafify_label(ft_label: str) -> str:
    """Convert fastText label back to human-readable."""
    return ft_label.replace("__label__", "").replace("_", " ")


def decompress_html(value: str) -> str:
    """Decompress base64+gzip encoded HTML if needed."""
    if not value or not isinstance(value, str):
        return ""
    value = value.strip()
    if not value:
        return ""
    if "<html" in value.lower() or "<body" in value.lower() or "<div" in value.lower():
        return value
    try:
        decoded = base64.b64decode(value)
        try:
            return gzip.decompress(decoded).decode("utf-8", errors="ignore")
        except Exception:
            return decoded.decode("utf-8", errors="ignore")
    except Exception:
        return value

## HTML Text Extraction

Extract meaningful content from raw HTML while filtering out noise.

In [ ]:
def extract_text_from_html(raw_html: str, max_body_chars: int = 4000) -> str:
    """
    Extract meaningful text from HTML for classification.
    
    Strategy:
    - Remove noisy tags (script, style, nav, footer, etc.)
    - Extract title, meta descriptions, Open Graph tags
    - Extract headings (h1, h2, h3) with weighting
    - Extract main body text
    """
    if not raw_html or not isinstance(raw_html, str):
        return ""

    tree = HTMLParser(raw_html)
    if tree is None:
        return ""

    # Remove noisy elements
    for tag in ["script", "style", "noscript", "svg", "canvas", "iframe", "footer", "nav", "form", "aside"]:
        for node in tree.css(tag):
            node.decompose()

    parts = []

    # Title (high weight)
    title = tree.css_first("title")
    if title:
        t = normalize_text(title.text())
        if t:
            parts.extend([t, t, t])

    # Meta description
    meta_desc = tree.css_first('meta[name="description"]')
    if meta_desc:
        v = normalize_text(meta_desc.attributes.get("content", ""))
        if v:
            parts.extend([v, v])

    # Open Graph tags
    og_title = tree.css_first('meta[property="og:title"]')
    if og_title:
        v = normalize_text(og_title.attributes.get("content", ""))
        if v:
            parts.extend([v, v])

    og_desc = tree.css_first('meta[property="og:description"]')
    if og_desc:
        v = normalize_text(og_desc.attributes.get("content", ""))
        if v:
            parts.append(v)

    # Headings
    for sel in ["h1", "h2", "h3"]:
        texts = []
        for node in tree.css(sel):
            txt = normalize_text(node.text())
            if txt:
                texts.append(txt)
        if texts:
            joined = " ".join(texts[:20])
            parts.extend([joined, joined])

    # Main body text
    body = tree.body
    if body:
        body_text = normalize_text(body.text(separator=" "))
        if body_text:
            parts.append(body_text[:max_body_chars])

    return normalize_text(" ".join(parts))


# Test extraction
test_html = """
<html>
  <head>
    <title>Best Football News & Scores | Premier League Updates</title>
    <meta name="description" content="Get the latest football news, match scores, and league standings." />
  </head>
  <body>
    <h1>Latest Football News</h1>
    <h2>Premier League Matchday Results</h2>
    <p>Today's matches brought exciting results across all divisions.</p>
    <script>console.log("noise");</script>
  </body>
</html>
"""

print("Extracted text:")
print(extract_text_from_html(test_html)[:500])

## Load Dataset

In [ ]:
def parse_label(value, category_lookup: dict) -> str | None:
    """Parse label from various formats."""
    if value is None:
        return None
    if isinstance(value, list):
        candidates = value
    elif isinstance(value, str):
        raw = value.strip()
        if not raw:
            return None
        if raw.startswith("["):
            try:
                parsed = json.loads(raw)
                candidates = parsed if isinstance(parsed, list) else [raw]
            except json.JSONDecodeError:
                candidates = re.split(r"[,;|]", raw)
        else:
            candidates = re.split(r"[,;|]", raw)
    else:
        candidates = [str(value)]

    for candidate in candidates:
        key = str(candidate).strip().lower()
        if not key:
            continue
        mapped = category_lookup.get(key)
        if mapped:
            return mapped
    return None


def load_dataset(path: Path, valid_categories: set[str]) -> pd.DataFrame:
    """Load dataset from CSV, JSONL, or Parquet."""
    suffix = path.suffix.lower()
    category_lookup = {c.lower(): c for c in valid_categories}

    if suffix == ".jsonl":
        records = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        df = pd.DataFrame(records)
    elif suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix == ".parquet":
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported format: {suffix}")

    # Handle different schemas
    if {"html", "label"}.issubset(df.columns):
        pass
    elif {"compressed_html", "categories"}.issubset(df.columns):
        df = df.copy()
        df["html"] = df["compressed_html"].apply(decompress_html)
        df["label"] = df["categories"].apply(lambda x: parse_label(x, category_lookup))

    # Clean data
    df = df.dropna(subset=["html", "label"]).copy()
    df["label"] = df["label"].apply(lambda x: parse_label(x, category_lookup))
    df = df.dropna(subset=["label"]).copy()

    # Validate
    bad_labels = sorted(set(df["label"]) - valid_categories)
    if bad_labels:
        print(f"Warning: Found unexpected labels: {bad_labels}")
        df = df[df["label"].isin(valid_categories)]

    return df


# Load your dataset
if DATA_PATH.exists():
    print(f"Loading from {DATA_PATH}...")
    df = load_dataset(DATA_PATH, set(CATEGORIES))
    print(f"Loaded {len(df)} samples")
    print(f"\nLabel distribution:\n{df['label'].value_counts()}")
else:
    print(f"Dataset not found at {DATA_PATH}")
    print("Create a dataset with 'html' and 'label' columns to proceed.")
    df = None

## Train Model

If you have a dataset loaded, run this cell to train the classifier.

In [ ]:
if df is not None and len(df) > 0:
    # Split data
    train_df, valid_df = train_test_split(
        df,
        test_size=0.15,
        random_state=42,
        stratify=df["label"],
    )
    print(f"Train: {len(train_df)}, Validation: {len(valid_df)}")

    # Prepare training files
    MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    train_txt = MODEL_OUTPUT_DIR / "train.ft.txt"
    valid_txt = MODEL_OUTPUT_DIR / "valid.ft.txt"

    def build_ft_line(html_text, label):
        text = extract_text_from_html(html_text)
        text = normalize_text(text.lower())
        return f"{safe_label(label, set(CATEGORIES))} {text}"

    # Write training data
    train_lines = [build_ft_line(row["html"], row["label"]) for _, row in tqdm(train_df.iterrows(), total=len(train_df))]
    valid_lines = [build_ft_line(row["html"], row["label"]) for _, row in tqdm(valid_df.iterrows(), total=len(valid_df))]

    train_txt.write_text("\n".join(train_lines), encoding="utf-8")
    valid_txt.write_text("\n".join(valid_lines), encoding="utf-8")

    # Train model
    print(f"\nTraining with epochs={TRAIN_EPOCHS}, dim={TRAIN_DIM}, lr={TRAIN_LR}...")
    model = fasttext.train_supervised(
        input=str(train_txt),
        lr=TRAIN_LR,
        epoch=TRAIN_EPOCHS,
        wordNgrams=2,
        dim=TRAIN_DIM,
        minn=2,
        maxn=5,
        bucket=2_000_000,
        loss="softmax",
        thread=os.cpu_count() or 4,
    )

    # Save model
    model.save_model(str(MODEL_PATH))
    print(f"Saved model: {MODEL_PATH}")

    # Evaluate
    result = model.test(str(valid_txt))
    print(f"\nValidation Results:")
    print(f"  Samples: {result[0]}")
    print(f"  Precision@1: {result[1]:.4f}")
    print(f"  Recall@1: {result[2]:.4f}")

    # Detailed evaluation
    y_true = valid_df["label"].tolist()
    y_pred = [unsafify_label(model.predict(extract_text_from_html(html).lower(), k=1)[0][0]) for html in tqdm(valid_df["html"].tolist())]

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    # Confusion matrix
    cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=CATEGORIES), index=CATEGORIES, columns=CATEGORIES)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.show()

    # Quantize
    print("\nQuantizing model...")
    quant_path = MODEL_OUTPUT_DIR / "website_classifier.ftz"
    model.quantize(input=str(train_txt), retrain=True, cutoff=50_000, thread=os.cpu_count() or 4)
    model.save_model(str(quant_path))
    print(f"Saved quantized model: {quant_path}")
else:
    print("No dataset loaded. Provide a dataset to train the model.")

## Inference

Use the trained model to classify new HTML pages.

In [ ]:
class WebsiteClassifier:
    """Simple wrapper for website classification."""

    def __init__(self, model_path: Path):
        self.model = fasttext.load_model(str(model_path))

    def predict(self, raw_html: str, k: int = 3) -> list[dict]:
        """Predict top-k categories for an HTML page."""
        text = extract_text_from_html(raw_html)
        text = normalize_text(text.lower())
        labels, probs = self.model.predict(text, k=k)
        return [{"label": unsafify_label(lbl), "score": float(prob)} for lbl, prob in zip(labels, probs)]

    def predict_batch(self, html_list: list[str], k: int = 3) -> list[list[dict]]:
        """Predict for multiple HTML pages."""
        texts = [normalize_text(extract_text_from_html(h).lower()) for h in html_list]
        labels, probs = self.model.predict(texts, k=k)
        return [[{"label": unsafify_label(lbl), "score": float(prob)} for lbl, prob in zip(lb, pr)] for lb, pr in zip(labels, probs)]


# Load model
if MODEL_PATH.exists():
    classifier = WebsiteClassifier(MODEL_PATH)
    print(f"Loaded model from {MODEL_PATH}")
else:
    print(f"Model not found at {MODEL_PATH}. Train first!")
    classifier = None

In [ ]:
if classifier:
    # Test examples
    examples = {
        "Sports": """
        <html>
          <head><title>Premier League Scores & Results</title></head>
          <body><h1>Football Match Results</h1><p>Latest scores and standings.</p></body>
        </html>
        """,
        "Shopping": """
        <html>
          <head><title>Best Laptop Deals - Buy Online</title></head>
          <body><h1>Shop Laptops</h1><p>Add to cart. Free shipping on orders over $50.</p></body>
        </html>
        """,
        "Health": """
        <html>
          <head><title>Diet Tips & Nutrition Advice</title></head>
          <body><h1>Healthy Eating Guide</h1><p>Learn about vitamins and balanced diets.</p></body>
        </html>
        """,
    }

    for expected, html in examples.items():
        preds = classifier.predict(html, k=3)
        print(f"Expected: {expected}")
        print(f"Predictions: {preds}")
        print()

## Command Line Usage

You can also use the CLI script for training and prediction:

```bash
# Train a model
python scripts/website_classifier.py train --data dataset.jsonl --output model.bin

# Predict single HTML
python scripts/website_classifier.py predict --html "<html>...</html>" --model model.bin

# Custom categories
python scripts/website_classifier.py train --data data.csv --output model.bin --categories "Tech,Business,Sports"
```

See `scripts/website_classifier.py` for full documentation.